<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Ingeniería y Ciencia de Datos</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Laboratorio de Procesamiento de Datos</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Módulo 1: Extracción de datos de diferentes fuentes: Archivos XML</h3>
    </div>
</div>


# Archivos XML

XML organiza la información como un árbol de elementos. Cada nodo puede tener una etiqueta, atributos, texto y nodos hijos. Esta jerarquía permite representar relaciones complejas, aunque requiere recorrer el árbol para llegar a los valores.

En los ejemplos se usan dos enfoques: `find`/`findall` para rutas conocidas y recorridos anidados cuando se necesita explorar la estructura. Al convertir XML a tabla hay que decidir qué nodo representa una fila y cómo tratar los elementos que faltan o se repiten.


El formato XML es común para el intercambio de datos estructurados. Python ofrece la librería estándar `xml.etree.ElementTree` para analizar y extraer información de archivos XML.

In [ ]:
# Configuración común para ejecutar esta sección de forma independiente
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dir_base = os.getcwd()
print(dir_base)
ruta = os.path.join(dir_base, 'Data')
print(ruta)

In [ ]:
import xml.etree.ElementTree as ET

In [ ]:
xml_data = '''
<personas>
  <persona>
    <nombre>Ana</nombre>
    <edad>23</edad>
  </persona>
  <persona>
    <nombre>Luis</nombre>
    <edad>31</edad>
  </persona>
</personas>
'''

In [ ]:
root = ET.fromstring(xml_data)
print('Personas extraídas del archivo XML:')
for persona in root.findall('persona'):
    nombre = persona.find('nombre').text
    edad = persona.find('edad').text
    print(f'Nombre: {nombre}, Edad: {edad}')

### Anatomía de un documento XML

Un XML es un **árbol**: todo cuelga de un único elemento raíz. Cada **nodo** puede tener:

- una **etiqueta** (`tag`), p. ej. `<country>`;
- **atributos** (`attrib`), p. ej. `name="Singapore"`;
- **texto** (`text`), p. ej. `<rank>4</rank>`;
- **nodos hijos** anidados.

```mermaid
flowchart LR
    D["data (raíz)"] --> C1["country name='Liechtenstein'"]
    D --> C2["country name='Singapore'"]
    C1 --> R1["rank: 1"]
    C1 --> Y1["year: 2008"]
    C1 --> G1["gdppc: 141100"]
```

Con `xml.etree.ElementTree` recorremos ese árbol a mano; pero pandas ofrece además un atajo muy potente: `pd.read_xml`.

#### Métodos clave para navegar el árbol

| Método / atributo | Qué hace |
|---|---|
| `getroot()` | devuelve el nodo raíz |
| `nodo.tag` | nombre de la etiqueta |
| `nodo.attrib` | diccionario de atributos |
| `nodo.text` | texto contenido |
| `nodo.find('hijo')` | primer hijo que coincide |
| `nodo.findall('hijo')` | lista de hijos que coinciden |
| `nodo.iter('tag')` | recorre todos los descendientes con esa etiqueta, sin importar el nivel |

In [ ]:
# Patrón muy común: recorrer el árbol y construir una lista de diccionarios
arbol = ET.parse(ruta + '/tabla_1.xml')
raiz = arbol.getroot()


In [ ]:
raiz

In [ ]:
#inspeccionar los datos del archivo xml
for nodo in raiz:
    print(nodo.attrib,nodo.text,nodo.tag)
    for sn in nodo:
        print(sn.attrib,sn.text,sn.tag)

In [ ]:
#Extraer los datos de tabla_1.xml
d={}
for nodo in raiz:
    d[nodo.tag]=[]
for nodo in raiz:
    d[nodo.tag].append(nodo.attrib['name'])
for nodo in raiz:
    for sn in nodo:
        d[sn.tag]=[]
for nodo in raiz:
    for sn in nodo:
        d[sn.tag].append(sn.text)
d

In [ ]:
pd.DataFrame(d)

In [ ]:
registros = []
for pais in raiz.findall('country'):
    registros.append({
        'pais': pais.attrib['name'],          # atributo del nodo
        'rank': int(pais.find('rank').text),  # texto de un hijo
        'year': int(pais.find('year').text),
        'gdppc': int(pais.find('gdppc').text)
    })

df_paises_xml = pd.DataFrame(registros)
df_paises_xml

##### Ejemplo con tabla_2.xml

In [ ]:
archivo_2=ET.parse(ruta+'/tabla_2.xml')
root=archivo_2.getroot()
for nodo in root:
    print(nodo.tag,nodo.attrib,nodo.text)

In [ ]:
for nodo in root:
    for subn in nodo:
        print(subn.tag,subn.attrib,subn.text)

In [ ]:
archivo=ET.parse(ruta+'/tabla_2.xml')
raiz=archivo.getroot()

In [ ]:
L=[]
for n in raiz.findall('documents/document'):
    d={}
    d[n.tag]=n.text
    for k,v in n.attrib.items():
        d[k]=v
    L.append(d)
pd.DataFrame(L)

#### Otro Ejemplo

In [ ]:
archivo='IFC-Subscriptions-and-Voting-Power-of-Member-Count.xml'
file=ET.parse(os.path.join(ruta, archivo))
root=file.getroot()

for nodo in root:
    for snodo in nodo:
        print(snodo.tag,snodo.attrib,snodo.text)
        for ssnodo in snodo:
            print(ssnodo.tag,ssnodo.attrib,ssnodo.text)

In [ ]:
# Lista para almacenar los datos
datos = []

# Iterar sobre cada fila en el XML
for row in root.findall(".//row"):
    datos_fila = {
        '_id': row.attrib.get('_id', ''),
        '_uuid': row.attrib.get('_uuid', ''),
        '_position': row.attrib.get('_position', ''),
        '_address': row.attrib.get('_address', ''),
        'member': row.findtext("member", ''),
        'amount_thousands_of_usd': row.findtext("amount_thousands_of_usd", ''),
        'percent_of_total_amount': row.findtext("percent_of_total_amount", ''),
        'number_of_votes': row.findtext("number_of_votes", ''),
        'percent_of_total_votes': row.findtext("percent_of_total_votes", ''),
        'as_of_date': row.findtext("as_of_date", '')
    }
    datos.append(datos_fila)

# Crear DataFrame
df = pd.DataFrame(datos)
df

#### El atajo moderno: `pd.read_xml`

Cuando el XML tiene una estructura regular (un nodo repetido por fila), `pd.read_xml` construye el `DataFrame` en **una sola línea**, detectando etiquetas hijas y atributos automáticamente. Hace lo mismo que el bucle anterior, pero sin escribirlo. (Requiere el paquete `lxml`.)

In [ ]:
# pd.read_xml hace en una línea lo que arriba hicimos a mano
pd.read_xml(ruta + '/tabla_1.xml')

#### Buscar en profundidad con `iter` y leer atributos

En árboles más profundos, `iter('tag')` encuentra **todos** los descendientes con esa etiqueta sin importar su nivel. En `tabla_2.xml` cada `<document>` guarda su texto en una sección `CDATA` y sus metadatos (clave, URL) en **atributos**.

In [ ]:
# Extraer cada documento con su clave, URL y un fragmento del texto
arbol2 = ET.parse(ruta + '/tabla_2.xml')
raiz2 = arbol2.getroot()

docs = []
for doc in raiz2.iter('document'):
    texto = (doc.text or '').strip()
    docs.append({
        'key': doc.attrib.get('KEY'),
        'web': doc.attrib.get('web'),
        'fragmento': texto[:50] + '...'
    })

df_docs = pd.DataFrame(docs)
df_docs

In [ ]:
# Los atributos también son datos: metadatos del autor (nodo raíz)
print('Atributos del autor :', raiz2.attrib)
print('Documentos encontrados:', len(list(raiz2.iter('document'))))

## Practica de Laboratorio: Extracción de Archivos XML

Utiliza el archivo `Data/plant_catalog.xml` para construir un pequeño catálogo de plantas en formato tabular (DataFrame). El XML contiene una colección de elementos `PLANT` y, dentro de cada uno, las etiquetas `COMMON`, `BOTANICAL`, `ZONE`, `LIGHT`, `PRICE` y `AVAILABILITY`. Extrae la información del XML, transformarla en un `DataFrame` y realiza consultas de interés, al final exporta el resultado a un archivo CSV.

#### Conteste las siguiente Preguntas

1. ¿Cuántas plantas contiene el catálogo?
2. ¿Cuántas plantas hay en cada zona?
3. ¿Qué tipo de iluminación aparece con mayor frecuencia?
4. ¿Qué nombre científico se repite más veces?


In [ ]:
# 1. Lee el archivo utilizando `xml.etree.ElementTree`.


In [ ]:
# 2. Identifica el elemento raíz y cuenta cuántas plantas contiene.


In [ ]:
# 3. Recorre cada elemento `PLANT` y extrae sus etiquetas.


In [ ]:
# 4. Construye un `DataFrame` con las columnas: `nombre_comun`,`nombre_cientifico`,`zona`,`luz`,`precio`,`disponibilidad`


In [ ]:
# 5. Exporta el resultado a `Data/plant_catalog.csv`.


In [ ]:
# 6. Lee nuevamente el CSV con `pd.read_csv()` y verifica que conserve el mismo número de registros.
